# Creating the dataset

This notebook will serve as the data set creation tool.
It is as simple as defining the glyphs that you want to include and then exporting them using the Glyph_exporter class as shown in the examples provided

In [ ]:
import mglyph as mg
import sys, os

## First try with simple_scaled_star

In [ ]:
import math
import random
import numpy as np
import json
import zipfile
import io
from datetime import datetime


def random_color():
    return f"#{random.randint(0, 0xFFFFFF):06x}"


def simple_scaled_star(x: float, canvas: mg.Canvas, color: str) -> None:
    
    canvas.tr.translate(0, mg.lerp(x, 0, 0.05))
    
    radius = mg.lerp(x, 0.01, canvas.ysize / 2)

    vertices = []
    for segment in range(5):
        vertices.append(mg.orbit(canvas.center, segment * 2 * math.pi / 5, radius))
        vertices.append(mg.orbit(canvas.center, (segment + 0.5) * 2 * math.pi / 5,
                         math.cos(2 * math.pi / 5) / math.cos(math.pi / 5) * radius))

    # Draw the star with a random outline color
    canvas.polygon(vertices, width='10p', linecap='round', color=color) 

# Initialize dataset metadata
dataset_info = {
    "name": "Experimental dataset with randomly colored stars",
    "time-of-creation": datetime.now().strftime("%Y-%m-%d"),
    "samples": []
}



data_zip = io.BytesIO()

with zipfile.ZipFile(data_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for i in range(5):
        color1 = random_color()
        color = random_color()

        while color1 == color:
            color = random_color()

        xvalues = np.random.uniform(0.0, 100.0, 120)

        folder_bytes = mg.export(
            lambda x, canvas: simple_scaled_star(x, canvas, color=color),  
            xvalues=xvalues,
            name=f"Random Colored Star {i+1}", short_name=f'star_{i+1}',
            path=None,
            author="Mohaned Anene BARKALLAH", email="Mohanedanene.barkallah@gmail.com", version="1.0.0"
        )

        
        with zipfile.ZipFile(folder_bytes, 'r') as temp_zip:
            
            metadata = None
            for file_name in temp_zip.namelist():
                if file_name.endswith(".json"):
                    metadata = json.loads(temp_zip.read(file_name).decode())
                    break
            
            if metadata:
                
                for file_name in temp_zip.namelist():
                    if file_name.endswith(".png"):
                        
                        new_file_name = f"Random Colored Star {i+1}-{file_name.split('-')[-1]}"
                        zipf.writestr(new_file_name, temp_zip.read(file_name))

                        
                        img_number = file_name.split('-')[-1].split('.')[0]
                        value = None
                        for img_data in metadata['images']:
                            if img_data[0] == f"{img_number.zfill(3)}.png":
                                value = img_data[1]
                                break
                        
                        if value is not None:
                            dataset_info["samples"].append({
                                "value": value,
                                "file": new_file_name
                            })


with zipfile.ZipFile(data_zip, 'a', zipfile.ZIP_DEFLATED) as zipf:
    zipf.writestr('_dataset-info.json', json.dumps(dataset_info, indent=4))


output_path = "data.zip"
with open(output_path, "wb") as f:
    f.write(data_zip.getvalue())

print(f"Dataset successfully saved to {output_path}")

## Generalized code for any glyph drawing function

In this section we will be trying to make glyphs in bulk with different parameters and zip them all together in one ZIP file that would act as the dataset that we want.

In [ ]:
import numpy as np
import json
import zipfile
import io
from datetime import datetime

def create_glyph_dataset(
    glyph_function,
    dataset_name,
    num_variants=5,
    samples_per_variant=120,
    value_range=(0.0, 100.0),
    output_path="data.zip",
    author_info=None
):
    """
    Creates a dataset ZIP file from any glyph-drawing function.
    
    Parameters:
    - glyph_function: Function that takes (x, canvas) and draws a glyph
    - dataset_name: Name for the dataset
    - num_variants: Number of glyph variants to generate
    - samples_per_variant: Number of samples per variant
    - value_range: Tuple of (min, max) for parameter values
    - output_path: Where to save the ZIP file
    - author_info: Dict with 'author', 'email', 'version' (optional)
    """
    # Default author
    author_info = author_info or {
        "author": "Unknown",
        "email": "",
        "version": "1.0.0"
    }

    dataset_info = {
        "name": dataset_name,
        "time-of-creation": datetime.now().isoformat(),
        "samples": []
    }

    data_zip = io.BytesIO()

    with zipfile.ZipFile(data_zip, 'w') as zipf:
        for i in range(num_variants):
            xvalues = np.random.uniform(*value_range, samples_per_variant)
            folder_bytes = mg.export(
                glyph_function,
                xvalues=xvalues,
                name=f"{dataset_name} {i+1}",
                short_name=f'glyph_{i+1}',
                path=None,
                **author_info
            )

            with zipfile.ZipFile(folder_bytes) as temp_zip:
                json_files = [f for f in temp_zip.namelist() if f.endswith('.json')]
                if json_files:
                    metadata = json.loads(temp_zip.read(json_files[0]).decode())
                    png_files = [f for f in temp_zip.namelist() if f.endswith('.png')]
                    for file_name in png_files:
                        img_num = file_name.split('-')[-1].split('.')[0]
                        new_name = f"{dataset_name} {i+1}-{file_name.split('-')[-1]}"
                        zipf.writestr(new_name, temp_zip.read(file_name))
                        target = f"{img_num.zfill(3)}.png"
                        value = next((img[1] for img in metadata['images'] if img[0] == target), None)
                        if value is not None:
                            dataset_info["samples"].append({"value": value, "file": new_name})

    with zipfile.ZipFile(data_zip, 'a') as zipf:
        zipf.writestr('_dataset-info.json', json.dumps(dataset_info, indent=4))

    with open(output_path, "wb") as f:
        f.write(data_zip.getvalue())

    return output_path


In [ ]:
# Usage Example:
xvalues=[0.1*x for x in range(1001)]
exporter = GlyphExporter("My Glyph Collection")

# Export individual glyphs (returns BytesIO but also stores internally)
exporter.export_glyph(blue_scaled_square, "Blue Square") #It can be simple the number of samples will be chosen randomly with random values
exporter.export_glyph(yellow_scaled_square, "yellow square", num_samples=5, value_range=(10.0, 90.0)) #you can set the number of glyphs you want and the value range
exporter.export_glyph(green_scaled_square, "Green square", num_samples=200, xvalues=xvalues) #you can chose the xvalues that you want, it overrides the num_samples if it is larger

# Final packaging
exporter.finalize_zip("data.zip")

The code above works fine however it is not optimal as it asks for extra parameters when calling it and giving mg.export() as the parameter to the function is better which would lead us to the next section below.

# Zipper class

In this section we will be creating the Zipper class by keeping the spirit of the previous code but getting mg.export() out of the function and use it as a parameter instead.

In [ ]:
import json
import zipfile
import mglyph as mg
import numpy as np
from datetime import datetime
import math
import random

class GlyphExporter:
    def __init__(self, dataset_name="Glyph Dataset"):
        """Initialize exporter with just zipping functionality"""
        self.dataset_name = dataset_name
        self.glyph_blobs = []  # Stores (variant_name, export_bytesio) pairs
        self.metadata = {
            "name": dataset_name,
            "time-of-creation": datetime.now().isoformat(),
            "samples": []
        }

    def add(self, export_result):

        self.glyph_blobs.append((export_result))
        return self  # Enable method chaining

    def finalize(self, output_path="glyphs.zip"):
        """Create final ZIP package with all glyphs"""
        with zipfile.ZipFile(output_path, 'w') as final_zip:
            for index, blob in enumerate(self.glyph_blobs):
                variant_name = f"sample_{index}"
                with zipfile.ZipFile(blob) as glyph_zip:
                    for file_name in glyph_zip.namelist():
                        if file_name.endswith('.png'):
                            new_name = f"{variant_name}-{file_name.split('-')[-1]}"
                            final_zip.writestr(new_name, glyph_zip.read(file_name))
                        elif file_name.endswith('.json'):
                            data = json.loads(glyph_zip.read(file_name).decode())
                            self.metadata['samples'].extend({
                                "value": img[1],
                                "file": f"{variant_name}-{img[0]}"
                            } for img in data['images'])


            final_zip.writestr('_dataset-info.json', json.dumps(self.metadata, indent=2))

        print(f"Dataset saved to {output_path}")
        return output_path

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is None:
            self.finalize()


In [ ]:
# example usage 1

with GlyphExporter("Color Squares") as exporter:
    # Metadata for export
    common_metadata = {
        "version": "1.0.0",
        "author": "Mohaned Anène Barkallah",
        "email": "mohaned@example.com",
        "author_public": True,
    }

    # Variant 1
    exporter.add(
        mg.export(
            blue_scaled_square,
            xvalues=np.random.uniform(0.0, 100.0, 50),
            name="Blue Squares",
            short_name="blue",
            path=None,
            **common_metadata
        )
    )

    # Variant 2
    custom_x = [0.1 * x for x in range(1001)]
    exporter.add(
        mg.export(
            green_scaled_square,
            xvalues=custom_x,
            name="Green Squares",
            short_name="green",
            path=None,
            **common_metadata
        )
    )

    # Variant 3
    exporter.add(
        mg.export(
            yellow_scaled_square,
            xvalues=np.linspace(10, 90, 20),
            name="Yellow Squares",
            short_name="yellow",
            path=None,
            silent=True,
            **common_metadata
        )
    )


In [ ]:
#Example usage 2
def random_color():
    return f"#{random.randint(0, 0xFFFFFF):06x}"

def simple_scaled_star(x: float, canvas: mg.Canvas) -> None:
    
    canvas.tr.translate(0, mg.lerp(x, 0, 0.05))
    
    radius = mg.lerp(x, 0.01, canvas.ysize / 2)

    vertices = []
    for segment in range(5):
        vertices.append(mg.orbit(canvas.center, segment * 2 * math.pi / 5, radius))
        vertices.append(mg.orbit(canvas.center, (segment + 0.5) * 2 * math.pi / 5,
                         math.cos(2 * math.pi / 5) / math.cos(math.pi / 5) * radius))

    # Draw the star with a random outline color
    canvas.polygon(vertices, width='10p', linecap='round', color=random_color()) 

with GlyphExporter('ColoredStars') as exporter:
    common_metadata = {
        "version": "1.0.0",
        "author": "Mohaned Anène Barkallah",
        "email": "mohaned@example.com",
        "author_public": True,
    }
    for _ in range(10):
        exporter.add(mg.export(simple_scaled_star,
                               xvalues=np.random.uniform(0.0, 100.0, 50),
                               path=None,
                               name=f"RCStars-{_}",
                               short_name=f"RC{_}",
                               **common_metadata,
                               silent=True)
        )